In [1]:
import pandas as pd
import numpy as np

# 1. Cargar el dataset
# Usamos keep_default_na=True para que identifique automáticamente celdas vacías
df = pd.read_csv('master_fra.csv', keep_default_na=True)
print("--- Estructura inicial ---")
print(df.info())

# 2. Limpieza de nombres de columnas
# Elimina espacios en blanco no deseados al inicio o final de los títulos
df.columns = df.columns.str.strip()

# 3. Limpieza y estandarización de columnas de texto
text_columns = ['CountryCode', 'target_group', 'subset', 'category', 'question_label', 'answer']
for col in text_columns:
    if col in df.columns:
        # Eliminar espacios extra y homogeneizar comillas raras si las hay
        df[col] = df[col].astype(str).str.strip()
        # Reemplazar valores que pandas lee como texto "nan" a un valor nulo real
        df[col] = df[col].replace(['nan', 'None', ''], np.nan)

# 4. Tratamiento de la columna 'percentage' (Porcentaje)
# A veces viene con caracteres como ':' o ':' seguidos de notas. Los convertimos a numérico.
if 'percentage' in df.columns:
    # Reemplazamos los dos puntos o valores no numéricos por NaN de forma segura
    df['percentage'] = pd.to_numeric(df['percentage'].astype(str).str.strip(), errors='coerce')

# 5. Tratamiento de la columna 'notes'
# En tus datos se observa que 'notes' tiene valores como "[1]" o "[0]". Limpiamos los corchetes si prefieres el número limpio.
if 'notes' in df.columns:
    df['notes'] = df['notes'].astype(str).str.extract(r'\[(\d+)\]')[0] # Extrae solo el número dentro de []
    df['notes'] = pd.to_numeric(df['notes'], errors='coerce') # Lo convierte a número (las celdas vacías serán NaN)

# 6. Eliminar duplicados exactos si existen
duplicados = df.duplicated().sum()
if duplicados > 0:
    print(f"\nEliminando {duplicados} filas duplicadas...")
    df.drop_duplicates(inplace=True)

# 7. Resumen de valores nulos tras la limpieza
print("\n--- Valores nulos por columna tras la limpieza ---")
print(df.isnull().sum())

# 8. Guardar el dataset limpio
df.to_csv('master_fra_limpio.csv', index=False)
print("\n¡Limpieza completada con éxito! Archivo guardado como 'master_fra_limpio.csv'")

--- Estructura inicial ---
<class 'pandas.DataFrame'>
RangeIndex: 270571 entries, 0 to 270570
Data columns (total 10 columns):
 #   Column          Non-Null Count   Dtype
---  ------          --------------   -----
 0   year            270571 non-null  int64
 1   CountryCode     270542 non-null  str  
 2   target_group    270571 non-null  str  
 3   subset          270542 non-null  str  
 4   category        270571 non-null  str  
 5   question_code   270542 non-null  str  
 6   question_label  270542 non-null  str  
 7   answer          265058 non-null  str  
 8   percentage      270542 non-null  str  
 9   notes           166584 non-null  str  
dtypes: int64(1), str(9)
memory usage: 20.6 MB
None

Eliminando 28 filas duplicadas...

--- Valores nulos por columna tras la limpieza ---
year                   0
CountryCode            1
target_group           0
subset                 1
category               0
question_code          1
question_label         1
answer              5485
percen